In [ ]:
#Importar as bibliotecas e as ferramentas que serão usadas;
import os
import pandas as pd
import panel as pn
import hvplot.pandas
import sqlalchemy
from sqlalchemy import create_engine, Column, String, ForeignKey, func, Integer
from sqlalchemy.orm import declarative_base, sessionmaker
from dotenv import load_dotenv

In [ ]:
#Carregar configurações do .env
load_dotenv() #Retorna True se o arquivo .env for encontrado

In [ ]:
#Inicialização e criação da base que as classes herdarão;
pn.extension()
Base = declarative_base()

In [ ]:
#Conexão dinâmica (Usando os dados do .env);
db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_name = os.getenv('DB_NAME')
db_port = os.getenv('DB_PORT', '5432')

In [ ]:
#Utilizei psycopg2 para conversar com o PostgreSQL.
DATABASE_URL = f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
session = Session()

In [ ]:
#Mapeamento das classes utilizando a Biblioteca SQLAlchemy;
class Usuario(Base):
    __tablename__ = 'usuario'
    login = Column(String, primary_key=True)

class Departamento(Base):
    __tablename__ = 'departamento'
    nome_dep = Column(String, primary_key=True)

class Medico(Base):
    __tablename__ = 'medico'
    id = Column(Integer, primary_key=True, autoincrement=True)
    crm = Column(String, primary_key=True)
    login = Column(String, ForeignKey('usuario.login'), primary_key=True)
    nome_dep = Column(String, ForeignKey('departamento.nome_dep'), nullable=False)

In [ ]:
#Widgets da Interface;
in_crm = pn.widgets.TextInput(name='CRM', placeholder='Ex: CE123456')
in_login = pn.widgets.Select(name='Selecionar Login do Usuário', options=[])
in_dep = pn.widgets.TextInput(name='Departamento', placeholder='Ex: Cardiologia')
input_busca = pn.widgets.TextInput(name='Buscar por CRM', placeholder='Pesquisar...')
alerta = pn.pane.Markdown("", styles={'color': 'red'})
tabela_visualizacao = pn.widgets.DataFrame(pd.DataFrame(), width=700, height=300)

In [ ]:
#Funções de Lógica e mensagens de confirmção;
def atualizar_menu_login():
    # Busca logins reais na tabela Usuario
    logins_existentes = [u.login for u in session.query(Usuario).all()]
    in_login.options = logins_existentes
    
def carregar_dados(event=None):
    atualizar_menu_login()
    query = session.query(Medico)
    if input_busca.value:
        query = query.filter(Medico.crm.ilike(f"%{input_busca.value}%"))
    df = pd.read_sql(query.statement, engine)
    tabela_visualizacao.value = df

def incluir_medico(event):
    try:
        #Verifica se os campos não estão vazios;
        if not in_crm.value or not in_login.value:
            alerta.object = "### Preencha CRM e Login!"
            return
            
        novo = Medico(crm=in_crm.value, login=in_login.value, nome_dep=in_dep.value)
        session.add(novo)
        session.commit()
        alerta.object = "### Médico incluído com sucesso!"
        carregar_dados()
    except Exception as e:
        session.rollback()
        alerta.object = f"### Erro: Verifique se o Login '{in_login.value}' e o Depto '{in_dep.value}' existem no banco."

def editar_medico(event):
    medico = session.query(Medico).filter_by(crm=in_crm.value, login=in_login.value).first()
    if medico:
        medico.nome_dep = in_dep.value
        session.commit()
        alerta.object = "### Departamento editado!"
        carregar_dados()
    else:
        alerta.object = "### Médico não encontrado (CRM + Login necessários)."

def remover_medico(event):
    medico = session.query(Medico).filter_by(crm=in_crm.value, login=in_login.value).first()
    if medico:
        session.delete(medico)
        session.commit()
        alerta.object = "### Médico removido!"
        carregar_dados()
    else:
        alerta.object = "### Não foi possível remover."

@pn.depends(input_busca)
def aba_grafico(event=None):
    res = session.query(Medico.nome_dep, func.count(Medico.crm)).group_by(Medico.nome_dep).all()
    if not res:
        return pn.pane.Markdown("### Sem dados para o gráfico.")
    
    df = pd.DataFrame(res, columns=['Departamentos', 'Total'])
    return df.hvplot.bar(
        x='Departamentos', y='Total', title="Médicos por Departamento",
        color='teal', rot=45, height=400, padding=0.4
    )


In [ ]:
#Nomes, proporções e botões;
btn_add = pn.widgets.Button(name='Incluir', button_type='success', width=150)
btn_upd = pn.widgets.Button(name='Editar', button_type='warning', width=150)
btn_del = pn.widgets.Button(name='Remover', button_type='danger', width=150)
btn_buscar = pn.widgets.Button(name='Buscar', button_type='primary', width=100, align='end')

btn_add.on_click(incluir_medico)
btn_upd.on_click(editar_medico)
btn_del.on_click(remover_medico)
btn_buscar.on_click(carregar_dados)
input_busca.param.watch(carregar_dados, 'value')

In [ ]:
#Eventos e Layout;
aba_gestao = pn.Column(
    "## 🏥 Gestão Hospitalar - Tabela Médico",
    pn.Row(in_crm, in_login, in_dep),
    pn.Row(btn_add, btn_upd, btn_del),
    alerta,
    pn.layout.Divider(),
    "### 🔎 Pesquisar Médico",
    pn.Row(input_busca, btn_buscar),
    pn.Spacer(height=30),
    tabela_visualizacao,
    margin=20
)

app = pn.Tabs(
    ("Gestão de Médicos", aba_gestao),
    ("Relatório", pn.Column("## 📊 Análise por Setor", aba_grafico, margin=20))
)

carregar_dados()
app.servable()